In [ ]:
# ==============================================================================
# SCRIPT DE CARTOGRAFÍA ESPACIAL: MAPAS DE RIESGO TOXICOLÓGICO ZMG
# ==============================================================================
# NOTA PARA EL COMITÉ EVALUADOR:
# Este script no altera los resultados toxicológicos calculados previamente.
# Su única función es tomar las bases de datos (.csv) y convertirlas en
# representaciones gráficas espaciales (mapas).

# --- IMPORTACIÓN DE HERRAMIENTAS (LIBRERÍAS) ---
import pandas as pd            # Herramienta para leer y manejar tablas de datos (funciona como un Excel en código).
import matplotlib.pyplot as plt # Herramienta principal para crear lienzos y dibujar gráficos.
import geopandas as gpd        # Versión geográfica de 'pandas'; asocia tablas de datos con coordenadas espaciales.
import numpy as np             # Herramienta para cálculos matemáticos avanzados (se usará para los logaritmos).
import osmnx as ox             # Herramienta que descarga los límites geográficos reales de los municipios de internet.
import matplotlib.patches as mpatches # Herramienta para dibujar los cuadritos de colores en la leyenda del mapa.
import warnings
warnings.filterwarnings('ignore') # Apaga las advertencias visuales del sistema para mantener la pantalla limpia.

# ==========================================
# 1. GENERACIÓN DEL MAPA BASE (FRONTERAS)
# ==========================================
print("Dibujando fronteras de los 9 municipios de la ZMG...")

# Se enlistan los 9 municipios con su nomenclatura oficial para que la herramienta los encuentre.
municipios_zmg = [
    'Guadalajara, Jalisco, Mexico', 'Zapopan, Jalisco, Mexico',
    'San Pedro Tlaquepaque, Jalisco, Mexico', 'Tonalá, Jalisco, Mexico',
    'Tlajomulco de Zúñiga, Jalisco, Mexico', 'El Salto, Jalisco, Mexico',
    'Ixtlahuacán de los Membrillos, Jalisco, Mexico', 'Juanacatlán, Jalisco, Mexico',
    'Zapotlanejo, Jalisco, Mexico'
]

# 'geocode_to_gdf' busca los nombres en OpenStreetMap y descarga matemáticamente
# el polígono (la silueta) de cada municipio para poder dibujarlo.
gdf_zmg = ox.geocode_to_gdf(municipios_zmg)

# Se crea una columna con nombres cortos solo para propósitos estéticos en las etiquetas del mapa.
gdf_zmg['nombre_corto'] = [
    'Guadalajara', 'Zapopan', 'Tlaquepaque', 'Tonalá', 'Tlajomulco',
    'El Salto', 'Ixtlahuacán', 'Juanacatlán', 'Zapotlanejo'
]

# Calculamos el "centroide" (el centro geométrico exacto) de cada municipio.
# Esto nos sirve para saber en qué coordenada exacta imprimir el nombre del municipio sin que quede chueco.
gdf_zmg['centroide'] = gdf_zmg.geometry.centroid


# ==========================================
# 2. MAPA DE RIESGO CARCINOGÉNICO (CÁNCER)
# ==========================================
print("Generando mapa de Riesgo Carcinogénico...")

# Se lee el archivo CSV que contiene el riesgo ya calculado con la metodología de L.B. Gratt.
df_cancer = pd.read_csv("Riesgo Cancer.csv")

# FUNCION CONDICIONAL DE COLOR (Similar a un "SI" en Excel)
# Asigna un color (verde, naranja, rojo) a cada industria dependiendo de su nivel de riesgo.
def asignar_color_cancer(riesgo):
    if riesgo <= 1e-6: return 'green'      # Aceptable
    elif riesgo <= 1e-4: return 'orange'   # Tolerable (requiere investigación)
    else: return 'red'                     # Inaceptable (requiere mitigación)

# Aplicamos la regla de color a toda la base de datos.
df_cancer['COLOR'] = df_cancer['RISK_TOTAL'].apply(asignar_color_cancer)

# TRATAMIENTO MATEMÁTICO: ESCALA LOGARÍTMICA PARA EL TAMAÑO DE LOS MARCADORES
# Justificación: Como hay empresas con riesgo de 0.00000000001 y otras con riesgo de 33,
# una relación lineal haría que una burbuja tape toda la ciudad y las demás sean invisibles invisibles.
# 'clip(lower=1e-12)' es un candado matemático para evitar intentar sacar el logaritmo de cero (lo cual daría error).
riesgo_log = np.log10(df_cancer['RISK_TOTAL'].clip(lower=1e-12))

# Normalizamos los valores (los comprimimos) para que la burbuja más chica mida 20 y la más grande 1000 píxeles.
riesgo_norm = (riesgo_log - riesgo_log.min()) / (riesgo_log.max() - riesgo_log.min() + 1e-9)
df_cancer['SIZE'] = riesgo_norm * 980 + 20

# Transformamos nuestra tabla normal a una tabla espacial.
# 'EPSG:4326' es el código internacional que le indica al programa que nuestras columnas
# LATITUD y LONGITUD son coordenadas GPS reales de la Tierra.
gdf_cancer = gpd.GeoDataFrame(
    df_cancer,
    geometry=gpd.points_from_xy(df_cancer.LONGITUD, df_cancer.LATITUD),
    crs="EPSG:4326"
)

# Preparamos un lienzo de dibujo de 12x10 pulgadas.
fig, ax = plt.subplots(figsize=(12, 10))

# Imprimimos la silueta de los municipios en color gris claro.
gdf_zmg.plot(ax=ax, color='#e0e0e0', edgecolor='white', linewidth=1.5)

# Este ciclo imprime los nombres de los municipios encima de sus respectivos centroides calculados antes.
for idx, row in gdf_zmg.iterrows():
    ax.annotate(text=row['nombre_corto'], xy=(row['centroide'].x, row['centroide'].y),
                horizontalalignment='center', fontsize=9, fontweight='bold',
                color='dimgrey', alpha=0.7)

# ORDEN DE CAPAS VISUALES:
# Ordenamos las burbujas de la más chica a la más grande. Así, al imprimir, los círculos pequeños
# se dibujan al final (hasta arriba) y no quedan ocultos detrás de los círculos masivos.
gdf_cancer = gdf_cancer.sort_values(by='SIZE', ascending=True)

# Imprimimos los marcadores industriales en el mapa.
# 'alpha=0.6' hace que los círculos sean semi-transparentes (como vidrio) para poder ver si hay industrias solapadas.
gdf_cancer.plot(ax=ax, color=gdf_cancer['COLOR'], markersize=gdf_cancer['SIZE'],
                edgecolor='black', linewidth=1, alpha=0.6)

# Configuramos títulos y apagamos los ejes (para que no parezca una gráfica con números a los lados, sino un mapa).
ax.set_title("Distribución Espacial del Riesgo Carcinogénico en la ZMG\n(Tamaño proporcional al riesgo)", fontsize=14, fontweight='bold')
ax.set_axis_off()

# Construimos la leyenda (los cuadritos de referencia) en la esquina inferior derecha.
green_patch = mpatches.Patch(color='green', label='Aceptable (<= 1x10^-6)')
orange_patch = mpatches.Patch(color='orange', label='Tolerable (1x10^-6 a 1x10^-4)')
red_patch = mpatches.Patch(color='red', label='Inaceptable (> 1x10^-4)')
plt.legend(handles=[green_patch, orange_patch, red_patch], loc='lower right', title="Nivel de Riesgo")

# Exportamos la imagen resultante en alta calidad (300 dpi) y recortamos bordes blancos innecesarios (bbox_inches='tight').
plt.savefig("Mapa_Vectorial_Cancer_Burbujas.png", dpi=300, bbox_inches='tight')
plt.close() # Cerramos el lienzo para no gastar memoria RAM.


# ==========================================
# 3. MAPA DE RIESGO CRÓNICO (NO CÁNCER)
# ==========================================
print("Generando mapa de Riesgo Crónico...")

# Se lee el archivo CSV correspondiente al Cociente de Peligro (Hazard Quotient - HQ).
df_nocancer = pd.read_csv("No Cancer.csv")

# FUNCION CONDICIONAL DE COLOR
# Todo HQ mayor a 1.0 representa un riesgo sistémico y se marca en rojo.
def asignar_color_nocancer(hq):
    if hq <= 1.0: return 'green'
    else: return 'red'

df_nocancer['COLOR'] = df_nocancer['NONCANCER_SCORE_TOTAL'].apply(asignar_color_nocancer)

# TRATAMIENTO MATEMÁTICO: ESCALA LOGARÍTMICA
# Se aplica el mismo principio para evitar que valores extremos distorsionen la visualización.
hq_log = np.log10(df_nocancer['NONCANCER_SCORE_TOTAL'].clip(lower=1e-5))
hq_norm = (hq_log - hq_log.min()) / (hq_log.max() - hq_log.min() + 1e-9)
df_nocancer['SIZE'] = hq_norm * 980 + 20

# Conversión de la tabla a formato espacial con coordenadas GPS (EPSG:4326).
gdf_nocancer = gpd.GeoDataFrame(
    df_nocancer,
    geometry=gpd.points_from_xy(df_nocancer.LONGITUD, df_nocancer.LATITUD),
    crs="EPSG:4326"
)

# Nuevo lienzo para el segundo mapa.
fig2, ax2 = plt.subplots(figsize=(12, 10))

# Se imprime nuevamente la silueta de los 9 municipios.
gdf_zmg.plot(ax=ax2, color='#e0e0e0', edgecolor='white', linewidth=1.5)

# Se imprimen las etiquetas de texto en los centroides.
for idx, row in gdf_zmg.iterrows():
    ax2.annotate(text=row['nombre_corto'], xy=(row['centroide'].x, row['centroide'].y),
                 horizontalalignment='center', fontsize=9, fontweight='bold',
                 color='dimgrey', alpha=0.7)

# Ordenamiento de capas visuales por tamaño (chicos arriba, grandes atrás).
gdf_nocancer = gdf_nocancer.sort_values(by='SIZE', ascending=True)

# Renderizado de los marcadores semi-transparentes (alpha=0.6).
gdf_nocancer.plot(ax=ax2, color=gdf_nocancer['COLOR'], markersize=gdf_nocancer['SIZE'],
                  edgecolor='black', linewidth=1, alpha=0.6)

# Títulos y formato limpio sin ejes cartesianos.
ax2.set_title("Distribución Espacial del Riesgo Crónico en la ZMG\n(Tamaño proporcional al Cociente de Peligro)", fontsize=14, fontweight='bold')
ax2.set_axis_off()

# Leyenda específica para los niveles del Cociente de Peligro (HQ).
green_patch2 = mpatches.Patch(color='green', label='Seguro (HQ <= 1.0)')
red_patch2 = mpatches.Patch(color='red', label='Riesgo Potencial para la Salud (HQ > 1.0)')
plt.legend(handles=[green_patch2, red_patch2], loc='lower right', title="Cociente de Peligro")

# Guardado final de la imagen.
plt.savefig("Mapa_Vectorial_NoCancer_Burbujas.png", dpi=300, bbox_inches='tight')
plt.close()

print("¡Mapas de burbujas proporcionales generados con éxito!")